# Import Library

In [6]:
import re
import random
import warnings
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import joblib

import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

warnings.filterwarnings("ignore")

# Set Random Seed

In [7]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Download NLTK Resource

In [8]:
try:
    nltk.download("punkt_tab")
except Exception:
    pass

try:
    nltk.download("punkt")
except Exception:
    pass

try:
    nltk.download("stopwords")
except Exception:
    pass

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Path dan Konfigurasi

In [9]:
TEXT_COL = "tweet"
LABEL_COL = "category"

SCENARIOS = [
    "10k",
    "20k",
    "30k",
    "40k"
]

PREPROCESS_ROOT = Path("dataset/preprocessed")
PREPROCESS_ROOT.mkdir(parents=True, exist_ok=True)

# Preprocessing

## Stemming & Stopword Removal

In [10]:
stopword_factory = StopWordRemoverFactory()

indonesian_stopwords = set(
    stopword_factory.get_stop_words()
)

extra_stopwords = {
    "rt","via","yg","dgn","nya",
    "nih","deh","dong","sih",
    "lah","kok","ya","ga",
    "gak","nggak","aja",
    "dll","dsb"
}

indonesian_stopwords |= extra_stopwords

stemmer_factory = StemmerFactory()

indo_stemmer = stemmer_factory.create_stemmer()

## Preprocessing Method

In [11]:
def clean_basic_text(text):

    text = str(text).lower()

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    text = re.sub(
        r"@\w+",
        " ",
        text
    )

    text = text.replace(
        "#",
        " "
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def preprocess_classical(text):

    text = clean_basic_text(text)

    tokens = word_tokenize(text)

    tokens = [
        t
        for t in tokens
        if t not in indonesian_stopwords
        and len(t) > 1
    ]

    tokens = [
        indo_stemmer.stem(t)
        for t in tokens
    ]

    tokens = [
        t
        for t in tokens
        if t.strip()
    ]

    return " ".join(tokens)


def preprocess_for_bert(text):

    text = str(text).lower()

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    text = re.sub(
        r"@\w+",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

# Do the loop

In [12]:
for scenario in SCENARIOS:

    print("="*60)
    print(f"PROCESSING {scenario}")
    print("="*60)

    DATA_PATH = Path(
        f"dataset/dataset{scenario}.csv"
    )

    OUTPUT_DIR = PREPROCESS_ROOT / scenario

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    ################################

    df = pd.read_csv(
        DATA_PATH
    )

    assert TEXT_COL in df.columns
    assert LABEL_COL in df.columns

    df = df.dropna(
        subset=[
            TEXT_COL,
            LABEL_COL
        ]
    ).copy()

    df[TEXT_COL] = df[
        TEXT_COL
    ].astype(str)

    df[LABEL_COL] = df[
        LABEL_COL
    ].astype(str)

    print(
        "Jumlah data:",
        len(df)
    )

    ################################

    le = LabelEncoder()

    df[
        "label_encoded"
    ] = le.fit_transform(
        df[LABEL_COL]
    )

    class_names = list(
        le.classes_
    )

    print(
        "Class names:",
        class_names
    )

    encoder_path = (
        PREPROCESS_ROOT /
        "label_encoder.pkl"
    )

    if not encoder_path.exists():

        joblib.dump(
            le,
            encoder_path
        )

    ################################

    df[
        "text_clean"
    ] = df[
        TEXT_COL
    ].apply(
        preprocess_classical
    )

    df[
        "text_bert"
    ] = df[
        TEXT_COL
    ].apply(
        preprocess_for_bert
    )

    ################################

    train_df, test_df = train_test_split(

        df,

        test_size=0.2,

        random_state=RANDOM_SEED,

        stratify=df[
            "label_encoded"
        ]

    )

    print(
        "Train:",
        len(train_df)
    )

    print(
        "Test:",
        len(test_df)
    )

    ################################

    X_train = train_df[
        "text_clean"
    ]

    X_test = test_df[
        "text_clean"
    ]

    y_train = train_df[
        "label_encoded"
    ]

    y_test = test_df[
        "label_encoded"
    ]

    ################################

    train_df_bert = train_df[
        [
            "text_bert",
            "label_encoded"
        ]
    ].rename(

        columns={

            "text_bert":
            "text",

            "label_encoded":
            "label"

        }

    )

    test_df_bert = test_df[
        [
            "text_bert",
            "label_encoded"
        ]
    ].rename(

        columns={

            "text_bert":
            "text",

            "label_encoded":
            "label"

        }

    )

    ################################

    X_train.to_csv(
        OUTPUT_DIR /
        "X_train.csv",
        index=False
    )

    X_test.to_csv(
        OUTPUT_DIR /
        "X_test.csv",
        index=False
    )

    y_train.to_csv(
        OUTPUT_DIR /
        "y_train.csv",
        index=False
    )

    y_test.to_csv(
        OUTPUT_DIR /
        "y_test.csv",
        index=False
    )

    ################################

    train_df_bert.to_csv(

        OUTPUT_DIR /
        "train_bert.csv",

        index=False

    )

    test_df_bert.to_csv(

        OUTPUT_DIR /
        "test_bert.csv",

        index=False

    )

    ################################

    print(
        f"{scenario} selesai."
    )

    print()

PROCESSING 10k
Jumlah data: 10000
Class names: ['Economy', 'Entertainment', 'Technology']
Train: 8000
Test: 2000
10k selesai.

PROCESSING 20k
Jumlah data: 20000
Class names: ['Economy', 'Entertainment', 'Technology']
Train: 16000
Test: 4000
20k selesai.

PROCESSING 30k
Jumlah data: 30000
Class names: ['Economy', 'Entertainment', 'Technology']
Train: 24000
Test: 6000
30k selesai.

PROCESSING 40k
Jumlah data: 40000
Class names: ['Economy', 'Entertainment', 'Technology']
Train: 32000
Test: 8000
40k selesai.



In [13]:
for scenario in SCENARIOS:

    print()
    print("="*50)
    print(scenario)
    print("="*50)

    folder = Path(
        f"dataset/preprocessed/{scenario}"
    )

    for file in sorted(
        folder.glob("*.csv")
    ):

        print(
            file.name
        )


10k
test_bert.csv
train_bert.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv

20k
test_bert.csv
train_bert.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv

30k
test_bert.csv
train_bert.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv

40k
test_bert.csv
train_bert.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv
